# ComfyUI × MiniMax H3 × Colab

一條龍：裝 ComfyUI → 備齊權重 → 啟動 → **把一句話編譯成正確的 H3 提示組** → TTS → 聲音驅動生成。

| 這份 notebook 解決什麼 | 怎麼解決 |
|---|---|
| 不知道該下哪些模型 | `h3colab config` 內建實查過的檔名與大小（官方 H3 / Turbo LoRA / PRISM） |
| 24GB 卡塞不下 | 依 GPU 自動選 profile：L4 用 `w6a8` UNET + 4-bit CLIP；A100 用 `int8` |
| 提示寫不好，AI 亂講話 | `prompt_compiler` 產出官方格式：風格句 + 時間碼 Storyboard + Audio 行 + 禁止項 |
| 沒有 TTS，不算聲音驅動 | Edge TTS 產出 32kHz wav，並用**真實音軌長度**回填時間碼，嘴型才對得上 |
| 掛著掛著就燒光算力 | 用完中斷 runtime；權重放 Drive 當倉庫 |

**執行順序**：下面每格依序按 Shift+Enter。權重下載最慢（30GB 上下），建議用 Google AI Pro 的額度跑。

In [ ]:
# == Cell 1：環境檢查 + 掛載 Drive ==
import os, sys, subprocess, json

os.environ.setdefault("PYTHONIOENCODING", "utf-8")

PROFILE = "l4"          # l4 / l4_ref / a100 / a100_ref / t4
STORE = "drive"         # drive = 雲端硬碟當倉庫（可重複用）；local = 直落本機（單次最快）
VOICE_DRIVEN = True     # 要做聲音驅動數字人 → 額外下載 ref2va 權重（PRISM / ref2va）

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive 已掛載")
except Exception as exc:
    print("掛不到 Drive（不在 Colab？）：", exc)

!pip -q install -U huggingface_hub requests
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install edge-tts

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, round(props.total_memory / 1024**3, 1), "GB",
          "SM" + f"{props.major}{props.minor}")

## Cell 2：寫入 h3colab 套件

下面幾格會把 `h3colab/` 整包寫進 runtime（提示編譯器、TTS、payload 建構、ComfyUI 客戶端、佈署腳本）。

In [ ]:
%%writefile h3colab/config.py
+"""設定：模型檔案來源、部署 profile、節點倉庫。
+
+權重來源（皆經 HuggingFace API 實查檔名與大小）：
+
+* 官方 H3 權重      ``Comfy-Org/MiniMax-H3``
+* Turbo LoRA       ``larryvrh/MiniMax-H3-Turbo-Lora``
+* PRISM（ref2va merge）``OliviaRossi/Minimax-h3_Singularity-x-10Eros-Max_PRISM``
+
+Profile 是「算力檔案」：一組 (GPU 級別 → 解析度 / 幀數 / 步數 / 權重精度 /
+節點選項)。Colab 上先 ``doctor`` 認出 GPU，再 ``setup --profile <key>``。
+"""
+
+from __future__ import annotations
+
+import os
+from dataclasses import dataclass
+
+# --- 路徑 -------------------------------------------------------------------
+
+# Colab 預設是 /content/ComfyUI；本機實測可設 H3COMFY_ROOT=D:\AI\ComfyUI
+# .strip() 是必要的：cmd 的 `set VAR=value && cmd` 會把尾端空白帶進值。
+COMFY_ROOT = os.environ.get("H3COMFY_ROOT", "/content/ComfyUI").strip()
+DRIVE_ROOT = os.environ.get("H3DRIVE_ROOT", "/content/drive/MyDrive/ComfyUI-H3").strip()
+
+# --- 倉庫 -------------------------------------------------------------------
+
+REPO_ORG = "Comfy-Org/MiniMax-H3"
+REPO_TURBO = "larryvrh/MiniMax-H3-Turbo-Lora"
+REPO_PRISM = "OliviaRossi/Minimax-h3_Singularity-x-10Eros-Max_PRISM"
+
+GIT_TURBO = "https://github.com/larryvrh/ComfyUI-MiniMax-H3-Turbo"
+GIT_SOL_ATTN = "https://github.com/Saganaki22/ComfyUI-sol-attn"
+GIT_DIRECTOR = "https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director"
+GIT_VIDEO_HELPER = "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite"
+
+
+@dataclass(frozen=True)
+class ModelFile:
+    """一個權重檔案：HF 來源 + 落地位置。"""
+
+    key: str
+    repo: str
+    filename: str
+    subdir: str
+    size_gb: float
+    task: str
+    note: str = ""
+
+    @property
+    def local_name(self) -> str:
+        return self.filename
+
+    @property
+    def local_path(self) -> str:
+        return f"{COMFY_ROOT}/models/{self.subdir}/{self.filename}"
+
+    @property
+    def drive_path(self) -> str:
+        return f"{DRIVE_ROOT}/models/{self.subdir}/{self.filename}"
+
+    @property
+    def hf_url(self) -> str:
+        return f"https://huggingface.co/{self.repo}/resolve/main/{self.filename}"
+
+
+# --- 權重清單 ---------------------------------------------------------------
+
+FL2VA_W6A8 = ModelFile(
+    "fl2va_w6a8", REPO_ORG, "minimax_h3_fl2va_pruned_w6a8.safetensors",
+    "diffusion_models", 14.89, "t2v/i2v/fl2v",
+    "pruned + w6a8，24GB 卡首選。",
+)
+FL2VA_INT8 = ModelFile(
+    "fl2va_int8", REPO_ORG, "minimax_h3_fl2va_pruned_int8_convrot.safetensors",
+    "diffusion_models", 19.53, "t2v/i2v/fl2v",
+    "本機 D:\\AI\\ComfyUI 用的就是這顆，品質基線。",
+)
+FL2VA_BF16 = ModelFile(
+    "fl2va_bf16", REPO_ORG, "minimax_h3_fl2va_pruned_bf16.safetensors",
+    "diffusion_models", 37.46, "t2v/i2v/fl2v",
+    "40GB 以下別用。",
+)
+REF2VA_W6A8 = ModelFile(
+    "ref2va_w6a8", REPO_ORG, "minimax_h3_ref2va_pruned_w6a8.safetensors",
+    "diffusion_models", 14.89, "r2v/v2v/rv2v",
+    "參考主體／參考音訊（聲音驅動）路徑。",
+)
+REF2VA_INT8 = ModelFile(
+    "ref2va_int8", REPO_ORG, "minimax_h3_ref2va_pruned_int8_convrot.safetensors",
+    "diffusion_models", 19.53, "r2v/v2v/rv2v",
+)
+PRISM_W6A8 = ModelFile(
+    "prism_w6a8", REPO_PRISM,
+    "Minimax-h3_Singularity_x_10Eros-Max_PRISM_w6a8.safetensors",
+    "diffusion_models", 14.81, "r2v/v2v/rv2v",
+    "Singularity x 10Eros-Max PRISM merge，ref2va。",
+)
+PRISM_INT8 = ModelFile(
+    "prism_int8", REPO_PRISM,
+    "Minimax-h3_Singularity_x_10Eros-Max_PRISM_int8.safetensors",
+    "diffusion_models", 18.89, "r2v/v2v/rv2v",
+)
+PRISM_FP16 = ModelFile(
+    "prism_fp16", REPO_PRISM,
+    "Minimax-h3_Singularity_x_10Eros-Max_PRISM_fp16.safetensors",
+    "diffusion_models", 61.70, "r2v/v2v/rv2v",
+    "太肥，Colab 別用。",
+)
+CLIP_NVFP4 = ModelFile(
+    "clip_nvfp4", REPO_ORG, "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
+    "text_encoders", 14.61, "all",
+    "Qwen3-VL 32B 4-bit；CLIPLoader 的 type 必須是 minimax。",
+)
+VIDEO_VAE_FP16 = ModelFile(
+    "video_vae_fp16", REPO_ORG, "minimax_h3_video_vae_fp16.safetensors",
+    "vae", 4.85, "all",
+)
+VIDEO_VAE_INT8 = ModelFile(
+    "video_vae_int8", REPO_ORG, "minimax_h3_video_vae_int8_convrot.safetensors",
+    "vae", 2.62, "all",
+    "省 2.2GB，24GB 卡用這個。",
+)
+AUDIO_VAE_FP32 = ModelFile(
+    "audio_vae_fp32", REPO_ORG, "minimax_h3_audio_vae_fp32.safetensors",
+    "vae", 0.56, "all",
+    "H3 原生立體聲；少了就沒聲音。",
+)
+TURBO_V4 = ModelFile(
+    "turbo_v4", REPO_TURBO, "minimax_h3_turbo_v4_step600_ema.safetensors",
+    "loras", 0.73, "fl2va",
+    "6–8 步成片；4 步 + 大動作會拖影。",
+)
+REF2V_TURBO = ModelFile(
+    "ref2v_turbo", REPO_ORG, "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
+    "loras", 1.82, "ref2va",
+    "ref2va 專用 turbo LoRA。",
+)
+
+ALL_MODELS: dict[str, ModelFile] = {
+    m.key: m
+    for m in (
+        FL2VA_W6A8, FL2VA_INT8, FL2VA_BF16,
+        REF2VA_W6A8, REF2VA_INT8,
+        PRISM_W6A8, PRISM_INT8, PRISM_FP16,
+        CLIP_NVFP4, VIDEO_VAE_FP16, VIDEO_VAE_INT8, AUDIO_VAE_FP32,
+        TURBO_V4, REF2V_TURBO,
+    )
+}
+
+# --- 算力檔案 ---------------------------------------------------------------
+
+@dataclass(frozen=True)
+class Profile:
+    """一組「GPU 級別 → 該怎麼跑」的參數。"""
+
+    key: str
+    label: str
+    vram_gb: int
+    width: int
+    height: int
+    length: int
+    steps: int
+    mode: str                # "official" = 官方 SigmaShift 流程；"turbo" = Turbo LoRA + Turbo Sampler
+    unet: ModelFile
+    clip: ModelFile
+    video_vae: ModelFile
+    audio_vae: ModelFile
+    turbo_lora: ModelFile | None = None
+    sol_attn: bool = False
+    ffn_chunks: int = 1
+    lowvram: bool = False
+    warn: str = ""
+    extra_models: tuple[ModelFile, ...] = ()
+
+    @property
+    def models(self) -> tuple[ModelFile, ...]:
+        items = [self.unet, self.clip, self.video_vae, self.audio_vae]
+        if self.turbo_lora is not None:
+            items.append(self.turbo_lora)
+        items.extend(self.extra_models)
+        return tuple(items)
+
+    @property
+    def megapixels(self) -> float:
+        return self.width * self.height / 1_000_000
+
+
+L4 = Profile(
+    key="l4",
+    label="L4 24GB · 0.4MP · 5 秒 · Turbo 8 步",
+    vram_gb=24,
+    width=864, height=480, length=124, steps=8,
+    mode="turbo",
+    unet=FL2VA_W6A8, clip=CLIP_NVFP4,
+    video_vae=VIDEO_VAE_INT8, audio_vae=AUDIO_VAE_FP32,
+    turbo_lora=TURBO_V4,
+    sol_attn=True, ffn_chunks=4, lowvram=True,
+    warn="L4 = SM89，sol-attn 官方支援；864x480 才是 0.4MP 草稿解析度。",
+)
+
+L4_REF = Profile(
+    key="l4_ref",
+    label="L4 24GB · 0.4MP · 5 秒 · PRISM(ref2va) 聲音驅動",
+    vram_gb=24,
+    width=864, height=480, length=124, steps=8,
+    mode="official",
+    unet=PRISM_W6A8, clip=CLIP_NVFP4,
+    video_vae=VIDEO_VAE_INT8, audio_vae=AUDIO_VAE_FP32,
+    sol_attn=True, ffn_chunks=4, lowvram=True,
+    warn="PRISM 是 ref2va merge：走 r2v + <Audio 1>，不能當 t2v/i2v 用。"
+         "它是 turbo merge，所以用 8 步（不是官方的 25 步）。",
+)
+
+L4_REF_TURBO = Profile(
+    key="l4_ref_turbo",
+    label="L4 24GB · 0.4MP · 5 秒 · 官方 ref2va + ref2v Turbo 4 步",
+    vram_gb=24,
+    width=864, height=480, length=124, steps=6,
+    mode="turbo",
+    unet=REF2VA_W6A8, clip=CLIP_NVFP4,
+    video_vae=VIDEO_VAE_INT8, audio_vae=AUDIO_VAE_FP32,
+    turbo_lora=REF2V_TURBO,
+    sol_attn=True, ffn_chunks=4, lowvram=True,
+    warn="這是本機 4090 已驗證的 ref2va + turbo 配方（4–6 步）。",
+)
+
+A100 = Profile(
+    key="a100",
+    label="A100 40GB · 1.0MP · 5 秒 · 官方 25 步",
+    vram_gb=40,
+    width=1344, height=768, length=124, steps=25,
+    mode="official",
+    unet=FL2VA_INT8, clip=CLIP_NVFP4,
+    video_vae=VIDEO_VAE_FP16, audio_vae=AUDIO_VAE_FP32,
+    sol_attn=True, ffn_chunks=2, lowvram=False,
+    warn="1344x768 是本機 4090 的正式接龍解析度；A100 才吃得下。",
+)
+
+A100_REF = Profile(
+    key="a100_ref",
+    label="A100 40GB · 1.0MP · 5 秒 · PRISM(ref2va) 聲音驅動",
+    vram_gb=40,
+    width=1344, height=768, length=124, steps=25,
+    mode="official",
+    unet=PRISM_INT8, clip=CLIP_NVFP4,
+    video_vae=VIDEO_VAE_FP16, audio_vae=AUDIO_VAE_FP32,
+    sol_attn=True, ffn_chunks=2, lowvram=False,
+)
+
+T4 = Profile(
+    key="t4",
+    label="T4 16GB · 0.4MP · 5 秒 · 慢（不建議）",
+    vram_gb=16,
+    width=864, height=480, length=124, steps=8,
+    mode="turbo",
+    unet=FL2VA_W6A8, clip=CLIP_NVFP4,
+    video_vae=VIDEO_VAE_INT8, audio_vae=AUDIO_VAE_FP32,
+    turbo_lora=TURBO_V4,
+    sol_attn=False, ffn_chunks=4, lowvram=True,
+    warn="T4 = SM75：sol-attn 不支援（要 SM86+），H3 走 bf16 還會很慢。"
+         "寧可花點數選 L4，也不要 T4。",
+)
+
+PROFILES: dict[str, Profile] = {
+    p.key: p for p in (L4, L4_REF, L4_REF_TURBO, A100, A100_REF, T4)
+}
+
+
+def pick_profile(vram_gb: float, *, needs_ref: bool = False) -> Profile:
+    """依 VRAM 挑一個合理 profile（needs_ref = 聲音驅動 / r2v 路徑）。"""
+    if vram_gb >= 38:
+        return A100_REF if needs_ref else A100
+    if vram_gb >= 22:
+        return L4_REF if needs_ref else L4
+    return T4
+
+
+# --- 任務 / 幀格 ------------------------------------------------------------
+
+TASK_TYPES = {
+    "t2v": "文生音視頻（fl2va UNET）",
+    "i2v": "圖生音視頻（fl2va UNET）",
+    "fl2v": "首尾幀生視頻（fl2va UNET）",
+    "r2v": "參考主體 + 參考音訊生視頻（ref2va UNET）← 聲音驅動數字人",
+}
+
+# MiniMax H3 的 17k+5 幀格：124 = 5.16s、243 = 10.1s、362 = 15.1s
+FRAME_GRID = tuple(17 * k + 5 for k in range(0, 22))
+
+
+def snap_length(frames: int) -> int:
+    """把任意幀數吸附到 H3 的 17k+5 網格（>= 目標值）。"""
+    if frames <= 5:
+        return 5
+    k = (frames - 5 + 16) // 17
+    return 17 * k + 5
+

In [ ]:
%%writefile h3colab/prompt_compiler.py
+"""提示強化：把「一句話」變成 MiniMax H3 真正吃得下的提示組。
+
+H3 用 Qwen3-VL 自由提示詞，但官方範例工作流的格式很固定：
+
+    Realistic live-action cinematic look: <場景>, <鏡頭>, <景深>, <色調> ...
+
+    Storyboard:
+    [0s-2s] <節拍 1：主體 + 動作 + 運鏡>
+    [2s-3.5s] <節拍 2>
+
+    Audio: <與畫面同步的具體聲音>
+    No text, logos or watermarks.
+
+多數人（包括很多「數字人」工作流）漏掉的就是這三塊：
+時間碼分鏡、音軌同步指示、禁止項。少了它們，H3 會自由發揮 → 亂講話、
+嘴型對對不上、場景漂移。
+
+本模組提供三種後端：
+
+* ``rule``    —— 純規則，不需網路。英文 brief 直接編譯；中文 brief 會明確報錯
+  （沒有 LLM 就別假裝能翻譯）。
+* ``llm``     —— 任何 OpenAI 相容端點（OpenRouter / Ollama / LM Studio）或
+  Gemini API。先用 LLM 寫出結構化草稿，再由本模組重新編排時間碼。
+* ``director``—— 直接呼叫本機 ComfyUI 的 MiniMax H3 Director 節點
+  ``POST /minimax/director/enhance``（用它內建的 Bernini 模板）。
+
+「聲音驅動」關鍵：編譯器會把 TTS 的**真實音軌長度**回填成時間碼，
+並在 Audio 行加上「嘴型對齊 <Audio 1>」的明示，H3 才知道要跟著聲音動。
+"""
+
+from __future__ import annotations
+
+import json
+import os
+import re
+from dataclasses import dataclass, field
+from typing import Any, Sequence
+
+CJK_RE = re.compile(r"[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]")
+
+
+def has_cjk(text: str) -> bool:
+    return bool(CJK_RE.search(text or ""))
+
+
+@dataclass
+class Beat:
+    """一個分鏡節拍。"""
+
+    start: float
+    end: float
+    action: str
+
+    def render(self) -> str:
+        return f"[{_fmt(self.start)}s-{_fmt(self.end)}s] {self.action.strip()}"
+
+
+@dataclass
+class Brief:
+    """一次生成的所有輸入。"""
+
+    subject: str = ""
+    scene: str = ""
+    beats: list[str] = field(default_factory=list)
+    style: str = "Realistic live-action cinematic look"
+    look: str = ""            # 鏡頭/景深/顆粒/色調等外觀細節
+    camera: str = ""          # 預設運鏡（會附在每個節拍後面）
+    audio: str | None = None  # None = 靜音
+    voice_driven: bool = False  # 嘴型對齊 <Audio 1>
+    duration: float = 5.0
+    negatives: str = "No text, logos or watermarks."
+    identity_lock: str = ""   # 填了就用 identity-lock 版式
+    template: str = "storyboard"   # "storyboard" | "identity_lock"
+    timed_beats: list[Beat] | None = None  # 已算好時間碼的節拍（優先於 beats）
+
+    def validate(self) -> None:
+        if self.template not in ("storyboard", "identity_lock"):
+            raise ValueError(f"未知 template: {self.template}")
+        if self.duration <= 0:
+            raise ValueError("duration 必須 > 0")
+
+
+def _fmt(value: float) -> str:
+    """0 -> '0'、2.0 -> '2'、3.5 -> '3.5'。"""
+    text = f"{value:.1f}".rstrip("0").rstrip(".")
+    return text or "0"
+
+
+# --- 時間碼 ------------------------------------------------------------------
+
+def distribute_beats(actions: Sequence[str], duration: float) -> list[Beat]:
+    """把節拍平均切在 duration 內（官方範例是等分的時間碼）。"""
+    actions = [a for a in actions if a and a.strip()]
+    if not actions:
+        return [Beat(0.0, duration, "")]
+    step = duration / len(actions)
+    beats: list[Beat] = []
+    for index, action in enumerate(actions):
+        start = round(index * step, 2)
+        end = round(duration if index == len(actions) - 1 else (index + 1) * step, 2)
+        beats.append(Beat(start, end, action))
+    return beats
+
+
+def retime_beats(beats: Sequence[Beat], durations: Sequence[float]) -> list[Beat]:
+    """依每句台詞的**真實音軌長度**重新切時間碼。
+
+    durations 與 beats 一一對應；長度不足時最後一段吃掉剩餘時間。
+    這是「聲音驅動」的核心：畫面時間碼必須跟語速一致，否則嘴型對不上。
+    """
+    if len(beats) != len(durations):
+        raise ValueError(
+            f"beats 與 durations 長度不符：{len(beats)} vs {len(durations)}"
+        )
+    out: list[Beat] = []
+    cursor = 0.0
+    for index, (beat, dur) in enumerate(zip(beats, durations)):
+        end = cursor + max(0.2, float(dur))
+        out.append(Beat(round(cursor, 2), round(end, 2), beat.action))
+        cursor = end
+    return out
+
+
+# --- 編譯 --------------------------------------------------------------------
+
+def compile_prompt(brief: Brief) -> str:
+    """把 Brief 編譯成 H3 提示組（純規則，不碰網路）。"""
+    brief.validate()
+    if brief.template == "identity_lock":
+        return _compile_identity_lock(brief)
+    return _compile_storyboard(brief)
+
+
+def _compile_storyboard(brief: Brief) -> str:
+    blocks: list[str] = []
+
+    head = brief.style.strip()
+    detail = brief.scene.strip()
+    if brief.subject.strip():
+        detail = f"{brief.subject.strip()}, {detail}" if detail else brief.subject.strip()
+    if brief.look.strip():
+        detail = f"{detail}, {brief.look.strip()}" if detail else brief.look.strip()
+    blocks.append(f"{head}: {detail}" if detail else head)
+
+    beats = brief.timed_beats or distribute_beats(
+        brief.beats or [brief.subject or brief.scene or "a slow continuous cinematic shot"],
+        brief.duration,
+    )
+    if brief.camera.strip():
+        beats = [
+            Beat(b.start, b.end, f"{b.action.strip()}, {brief.camera.strip()}")
+            for b in beats
+        ]
+    blocks.append("Storyboard:\n" + "\n".join(b.render() for b in beats))
+
+    if brief.voice_driven:
+        audio = (brief.audio or "clear spoken voice").strip()
+        blocks.append(
+            f"Audio: {audio}, spoken from <Audio 1>, lip movements and jaw "
+            f"synchronized to <Audio 1>, steady face, no extra talking."
+        )
+    elif brief.audio:
+        blocks.append(f"Audio: {brief.audio.strip()}")
+    else:
+        blocks.append("No audio - silent video.")
+
+    if brief.negatives.strip():
+        blocks.append(brief.negatives.strip())
+    return "\n\n".join(blocks)
+
+
+def _compile_identity_lock(brief: Brief) -> str:
+    """本機 skill 用的版式：IDENTITY LOCK + [Shot n]（接龍風格）。"""
+    header_parts = ["Photoreal cinematic", brief.scene.strip(), brief.look.strip()]
+    header = ", ".join(part for part in header_parts if part) + ", 16:9, 24fps."
+    lines: list[str] = [header]
+    if brief.identity_lock.strip():
+        lines.append(f"IDENTITY LOCK: {brief.identity_lock.strip()}")
+        lines.append(
+            "Same subject, same outfit, same location landmarks, "
+            "same light direction entire clip."
+        )
+    else:
+        lines.append("Same subject, same outfit, same setting entire clip.")
+
+    beats = distribute_beats(brief.beats or [brief.subject], brief.duration)
+    for index, beat in enumerate(beats, start=1):
+        action = beat.action.strip()
+        if index > 1:
+            action = (
+                "Continuous with previous clip. Start exactly from the previous "
+                f"ending pose. Do not reset pose. {action}"
+            )
+        if brief.camera.strip():
+            action = f"{action}, {brief.camera.strip()}"
+        lines.append(f"[Shot {index}] {action}")
+
+    lines.append(
+        "Motion rules: grounded feet, no face morph, no outfit change, "
+        "no extra people, no text, no logo, no subtitles."
+    )
+    if brief.voice_driven:
+        lines.append(
+            "Audio: spoken from <Audio 1>, lip movements synchronized to <Audio 1>."
+        )
+    elif brief.audio:
+        lines.append(f"Audio: {brief.audio.strip()}")
+    else:
+        lines.append("No audio - silent video.")
+    return "\n\n".join(lines)
+
+
+# --- LLM 後端 ---------------------------------------------------------------
+
+H3_JSON_DIRECTIVE = """You write production prompts for MiniMax H3, a model that
+generates video and synchronized audio in one pass.
+
+Return ONLY a JSON object with exactly these keys:
+{
+  "style_sentence": "one sentence: <style>: <subject>, <setting>, <lens>, <depth of field>, <grain>, <palette>",
+  "beats": ["one concrete visible beat per element: subject + action + camera, one sentence each"],
+  "audio": "what is heard in sync with the picture (ambient sounds + what is said). Empty string if silent.",
+  "subject": "short appearance description of the main subject",
+  "scene": "short description of location and lighting"
+}
+
+Rules:
+- English only, no Chinese characters.
+- 2 to 4 beats. Each beat is one visible action, no emotional or literary prose.
+- Never add subjects the brief did not mention.
+- If the brief is Chinese, translate the intent faithfully and keep proper names in both languages.
+- If the brief has someone speaking, put the exact spoken line in "audio".
+- No markdown fences, no commentary.
+"""
+
+
+@dataclass
+class LLMConfig:
+    """LLM 設定。provider = "openai"（OpenAI 相容）或 "gemini"。"""
+
+    provider: str = "openai"
+    model: str = ""
+    base_url: str = "https://openrouter.ai/api/v1"
+    api_key: str = ""
+    timeout: int = 180
+
+    @classmethod
+    def from_env(cls) -> "LLMConfig":
+        """從環境變數組出設定（Colab 通常用 GOOGLE_API_KEY 或 OPENROUTER_API_KEY）。"""
+        google = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
+        if google:
+            return cls(
+                provider="gemini",
+                model=os.environ.get("H3_LLM_MODEL", "gemini-2.5-flash"),
+                api_key=google,
+            )
+        openrouter = os.environ.get("OPENROUTER_API_KEY")
+        if openrouter:
+            return cls(
+                provider="openai",
+                model=os.environ.get("H3_LLM_MODEL", "google/gemini-2.5-flash"),
+                base_url="https://openrouter.ai/api/v1",
+                api_key=openrouter,
+            )
+        ollama = os.environ.get("OLLAMA_URL")
+        if ollama:
+            return cls(
+                provider="openai",
+                model=os.environ.get("H3_LLM_MODEL", "qwen3:8b"),
+                base_url=f"{ollama.rstrip('/')}/v1",
+                api_key="ollama",
+            )
+        raise RuntimeError(
+            "找不到 LLM 設定：請設 GOOGLE_API_KEY / GEMINI_API_KEY / "
+            "OPENROUTER_API_KEY / OLLAMA_URL 其中一個"
+        )
+
+
+def _post_json(url: str, payload: dict, headers: dict, timeout: int) -> dict:
+    import requests
+
+    response = requests.post(url, json=payload, headers=headers, timeout=timeout)
+    if response.status_code >= 400:
+        raise RuntimeError(f"LLM HTTP {response.status_code}: {response.text[:500]}")
+    return response.json()
+
+
+def llm_complete(config: LLMConfig, user_text: str) -> str:
+    """呼叫 LLM 拿純文字回應。"""
+    if config.provider == "gemini":
+        url = (
+            "https://generativelanguage.googleapis.com/v1beta/models/"
+            f"{config.model}:generateContent?key={config.api_key}"
+        )
+        payload = {
+            "contents": [{"parts": [{"text": user_text}]}],
+            "generationConfig": {"temperature": 0.7, "maxOutputTokens": 1200},
+        }
+        data = _post_json(url, payload, {"Content-Type": "application/json"}, config.timeout)
+        parts = data.get("candidates", [{}])[0].get("content", {}).get("parts", [])
+        return "".join(part.get("text", "") for part in parts)
+
+    url = f"{config.base_url.rstrip('/')}/chat/completions"
+    headers = {"Content-Type": "application/json"}
+    if config.api_key:
+        headers["Authorization"] = f"Bearer {config.api_key}"
+    payload = {
+        "model": config.model,
+        "messages": [{"role": "user", "content": user_text}],
+        "temperature": 0.7,
+        "max_tokens": 1200,
+    }
+    data = _post_json(url, payload, headers, config.timeout)
+    return data["choices"][0]["message"]["content"]
+
+
+def llm_json(config: LLMConfig, user_text: str) -> dict:
+    """呼叫 LLM 並解析 JSON（容忍 ```json 包裹與前後雜訊）。"""
+    raw = llm_complete(config, user_text).strip()
+    cleaned = raw
+    if cleaned.startswith("```"):
+        cleaned = cleaned.split("```")[1]
+        if cleaned.lstrip().lower().startswith("json"):
+            cleaned = cleaned.lstrip()[4:]
+    cleaned = cleaned.strip()
+    start = cleaned.find("{")
+    end = cleaned.rfind("}")
+    if start == -1 or end == -1:
+        raise RuntimeError(f"LLM 沒有回傳 JSON：{raw[:400]}")
+    return json.loads(cleaned[start:end + 1])
+
+
+def _brief_to_text(brief: Brief) -> str:
+    return " | ".join(
+        part for part in (
+            brief.subject, brief.scene, brief.look, brief.camera,
+            " ".join(brief.beats), brief.audio or "",
+        ) if part
+    )
+
+
+def enhance_with_llm(brief: Brief, config: LLMConfig) -> Brief:
+    """用 LLM 把 brief 強化成結構化 Brief（英文）。"""
+    payload = llm_json(config, f"{H3_JSON_DIRECTIVE}\n\nBrief: {_brief_to_text(brief)}")
+    beats = payload.get("beats") or []
+    if isinstance(beats, str):
+        beats = [beats]
+    # 已經用 TTS 算好時間碼時，節拍數不能被 LLM 改掉（否則嘴型對不上）
+    keep_timing = brief.timed_beats is not None
+    return Brief(
+        subject=payload.get("subject") or brief.subject,
+        scene=payload.get("scene") or brief.scene,
+        beats=list(brief.beats) if keep_timing else ([str(b) for b in beats] or list(brief.beats)),
+        style=brief.style,
+        look=payload.get("style_sentence") or brief.look,
+        camera=brief.camera,
+        audio=payload.get("audio") or brief.audio,
+        voice_driven=brief.voice_driven,
+        duration=brief.duration,
+        negatives=brief.negatives,
+        identity_lock=brief.identity_lock,
+        template=brief.template,
+        timed_beats=brief.timed_beats,
+    )
+
+
+# --- Director 後端 ----------------------------------------------------------
+
+def enhance_with_director(
+    brief: Brief,
+    base_url: str,
+    task_type: str = "t2v",
+    llm_config: LLMConfig | None = None,
+) -> str:
+    """呼叫 ComfyUI 的 MiniMax H3 Director 提示強化路由。
+
+    需要 ComfyUI 裝了 ComfyUI_MiniMaxH3_Director，且該節點有可用的 LLM。
+    """
+    import requests
+
+    llm_config = llm_config or LLMConfig.from_env()
+    response = requests.post(
+        f"{base_url.rstrip('/')}/minimax/director/enhance",
+        json={
+            "prompt": _brief_to_text(brief),
+            "task_type": task_type,
+            "model": llm_config.model,
+            "llm_url": "" if llm_config.provider == "gemini" else llm_config.base_url,
+            "api_format": "openai",
+            "api_key": llm_config.api_key,
+            "output_language": "English",
+        },
+        timeout=llm_config.timeout,
+    )
+    if response.status_code >= 400:
+        raise RuntimeError(
+            f"Director enhance HTTP {response.status_code}: {response.text[:400]}"
+        )
+    data = response.json()
+    text = data.get("text") or data.get("prompt") or data.get("enhanced")
+    if not text:
+        raise RuntimeError(f"Director 沒有回傳提示詞：{json.dumps(data)[:400]}")
+    return text
+
+
+# --- 統一入口 ---------------------------------------------------------------
+
+def enhance(
+    brief: Brief,
+    backend: str = "rule",
+    llm_config: LLMConfig | None = None,
+    base_url: str = "http://127.0.0.1:8188",
+    task_type: str = "t2v",
+) -> str:
+    """把 Brief 變成最終 H3 提示組。
+
+    backend: "rule"（離線編譯）/ "llm"（LLM 強化後再編譯）/ "director"（Director 節點）。
+    """
+    if backend == "rule":
+        if has_cjk(_brief_to_text(brief)):
+            raise ValueError(
+                "純規則編譯器不做翻譯：中文 brief 請改用 backend='llm' 或 "
+                "backend='director'，或直接給英文 beats。"
+            )
+        return compile_prompt(brief)
+    if backend == "llm":
+        llm_config = llm_config or LLMConfig.from_env()
+        return compile_prompt(enhance_with_llm(brief, llm_config))
+    if backend == "director":
+        return enhance_with_director(brief, base_url, task_type, llm_config)
+    raise ValueError(f"未知 backend: {backend}")
+
+
+

In [ ]:
%%writefile h3colab/payloads.py
+"""ComfyUI API 圖建構：把規格翻成可直接 POST /prompt 的節點圖。
+
+節點接線依據（全部對過真實 ComfyUI /object_info）：
+
+* 官方 25 步路徑：``UNETLoader → MiniMaxH3SigmaShift(12,3) → BasicScheduler(simple)
+  → BasicGuider → SamplerCustomAdvanced``
+* Turbo 路徑：``UNETLoader → MiniMaxH3TurboLoRA → BasicGuider/BasicScheduler
+  + MiniMaxH3TurboSampler``（**不含** SigmaShift — 官方 Turbo 範例工作流與本機
+  已驗證 payload 都是這樣）
+* 條件化：``MiniMaxH3ImageToVideo``（t2v/i2v/fl2v）或
+  ``MiniMaxH3ReferenceToVideo``（r2v，音訊走 ``ref_audio_N`` 槽）
+
+選配：sol-attn 的稀疏注意力 / FFN 分塊（節點沒裝就自動略過）。
+"""
+
+from __future__ import annotations
+
+from dataclasses import dataclass, field
+
+from .config import snap_length
+
+
+@dataclass
+class GenSpec:
+    """一次生成的完整規格。"""
+
+    task: str = "t2v"                 # t2v | i2v | fl2v | r2v
+    prompt: str = ""
+    width: int = 864
+    height: int = 480
+    length: int = 124
+    fps: float = 24.0
+    seed: int = 42
+    steps: int = 8
+    mode: str = "turbo"               # turbo | official
+    sampler: str = "res_multistep"
+    scheduler: str = "simple"
+    sigma_shift: bool | None = None   # None = 依 mode 自動（turbo 不加、official 加）
+    shift_video: float = 12.0
+    shift_audio: float = 3.0
+    unet: str = ""
+    clip: str = ""
+    video_vae: str = ""
+    audio_vae: str = ""
+    turbo_lora: str | None = None
+    turbo_lowvram: bool = True
+    sol_attn: bool = False
+    sol_tau: float = 1.3
+    sol_min_tokens: int = 4096
+    sol_thresh: str = "diag"
+    ffn_chunks: int = 1
+    ffn_min_tokens: int = 8192
+    first_frame: str | None = None
+    last_frame: str | None = None
+    ref_images: list[str] = field(default_factory=list)
+    ref_audios: list[str] = field(default_factory=list)
+    ref_videos: list[str] = field(default_factory=list)
+    ref_image_size: str = "match"
+    filename_prefix: str = "video/h3"
+    video_format: str = "auto"
+    video_codec: str = "auto"
+    bit_depth: int = 8
+
+    def resolved_length(self) -> int:
+        return snap_length(int(self.length))
+
+
+class _Graph:
+    """依序配號的節點圖。"""
+
+    def __init__(self) -> None:
+        self.nodes: dict[str, dict] = {}
+        self._counter = 0
+
+    def add(self, class_type: str, inputs: dict) -> str:
+        self._counter += 1
+        node_id = str(self._counter)
+        self.nodes[node_id] = {"class_type": class_type, "inputs": inputs}
+        return node_id
+
+
+def build_graph(
+    spec: GenSpec,
+    available: set[str] | None = None,
+) -> tuple[dict, list[str]]:
+    """建出 API 格式的節點圖。
+
+    available = 伺服器上實際存在的節點類名；給了才知道選配節點能不能用。
+    回傳 (graph, notes)。
+    """
+    notes: list[str] = []
+    graph = _Graph()
+    have = available
+
+    def supports(class_type: str) -> bool:
+        return have is None or class_type in have
+
+    # --- 模型 ---------------------------------------------------------------
+    unet_id = graph.add("UNETLoader", {
+        "unet_name": spec.unet,
+        "weight_dtype": "default",
+    })
+    clip_id = graph.add("CLIPLoader", {
+        "clip_name": spec.clip,
+        "type": "minimax",
+        "device": "default",
+    })
+    video_vae_id = graph.add("VAELoader", {"vae_name": spec.video_vae})
+    audio_vae_id = graph.add("VAELoader", {"vae_name": spec.audio_vae})
+
+    model_id = unet_id
+
+    use_shift = spec.sigma_shift
+    if use_shift is None:
+        use_shift = spec.mode != "turbo"
+    if use_shift:
+        model_id = graph.add("MiniMaxH3SigmaShift", {
+            "model": [model_id, 0],
+            "shift_video": spec.shift_video,
+            "shift_audio": spec.shift_audio,
+        })
+
+    # --- 選配：sol-attn / FFN 分塊 / Turbo LoRA -----------------------------
+    if spec.ffn_chunks and spec.ffn_chunks > 1:
+        if supports("MiniMaxH3ChunkFeedForward"):
+            model_id = graph.add("MiniMaxH3ChunkFeedForward", {
+                "model": [model_id, 0],
+                "enabled": True,
+                "chunks": int(spec.ffn_chunks),
+                "min_tokens": int(spec.ffn_min_tokens),
+            })
+        else:
+            notes.append("MiniMaxH3ChunkFeedForward 不在伺服器上，略過 FFN 分塊。")
+
+    if spec.sol_attn:
+        if supports("MiniMaxH3MemoryEfficientSolAttentionPatch"):
+            model_id = graph.add("MiniMaxH3MemoryEfficientSolAttentionPatch", {
+                "model": [model_id, 0],
+                "enabled": True,
+                "tau": float(spec.sol_tau),
+                "min_tokens": int(spec.sol_min_tokens),
+                "strict": False,
+                "thresh_type": spec.sol_thresh,
+                "int8_qk": False,
+                "int8_pv": False,
+            })
+        elif supports("SolAttentionPatch"):
+            model_id = graph.add("SolAttentionPatch", {
+                "model": [model_id, 0],
+                "enabled": True,
+                "tau": float(spec.sol_tau),
+                "min_tokens": int(spec.sol_min_tokens),
+                "strict": False,
+                "thresh_type": spec.sol_thresh,
+                "int8_qk": False,
+                "int8_pv": False,
+            })
+        else:
+            notes.append("sol-attn 節點不在伺服器上，改用預設注意力（速度會慢一些）。")
+
+    if spec.mode == "turbo":
+        if not spec.turbo_lora:
+            raise ValueError("mode='turbo' 需要 turbo_lora")
+        if supports("MiniMaxH3TurboLoRA"):
+            model_id = graph.add("MiniMaxH3TurboLoRA", {
+                "model": [model_id, 0],
+                "lora_name": spec.turbo_lora,
+                "strength": 1.0,
+                "low_vram": bool(spec.turbo_lowvram),
+            })
+        else:
+            notes.append(
+                "MiniMaxH3TurboLoRA 不在伺服器上，改用官方 25 步流程（慢很多）。"
+            )
+
+    # --- 條件化 -------------------------------------------------------------
+    length = spec.resolved_length()
+    if spec.task == "r2v":
+        cond_inputs: dict = {
+            "clip": [clip_id, 0],
+            "vae": [video_vae_id, 0],
+            "audio_vae": [audio_vae_id, 0],
+            "prompt": spec.prompt,
+            "width": int(spec.width),
+            "height": int(spec.height),
+            "length": length,
+            "ref_image_size": spec.ref_image_size,
+        }
+        for index, name in enumerate(spec.ref_images):
+            cond_inputs[f"ref_image_{index}"] = name
+        for index, name in enumerate(spec.ref_audios):
+            cond_inputs[f"ref_audio_{index}"] = name
+        for index, name in enumerate(spec.ref_videos):
+            cond_inputs[f"ref_video_{index}"] = name
+        cond_id = graph.add("MiniMaxH3ReferenceToVideo", cond_inputs)
+    else:
+        cond_inputs = {
+            "clip": [clip_id, 0],
+            "vae": [video_vae_id, 0],
+            "prompt": spec.prompt,
+            "width": int(spec.width),
+            "height": int(spec.height),
+            "length": length,
+        }
+        if spec.first_frame:
+            cond_inputs["first_frame"] = spec.first_frame
+        if spec.last_frame:
+            cond_inputs["last_frame"] = spec.last_frame
+        cond_id = graph.add("MiniMaxH3ImageToVideo", cond_inputs)
+
+    # --- 採樣 ---------------------------------------------------------------
+    scheduler_id = graph.add("BasicScheduler", {
+        "model": [model_id, 0],
+        "scheduler": spec.scheduler,
+        "steps": int(spec.steps),
+        "denoise": 1.0,
+    })
+    guider_id = graph.add("BasicGuider", {
+        "model": [model_id, 0],
+        "conditioning": [cond_id, 0],
+    })
+    if spec.mode == "turbo" and supports("MiniMaxH3TurboSampler"):
+        sampler_id = graph.add("MiniMaxH3TurboSampler", {})
+    else:
+        sampler_id = graph.add("KSamplerSelect", {"sampler_name": spec.sampler})
+    noise_id = graph.add("RandomNoise", {"noise_seed": int(spec.seed)})
+    sampled_id = graph.add("SamplerCustomAdvanced", {
+        "noise": [noise_id, 0],
+        "guider": [guider_id, 0],
+        "sampler": [sampler_id, 0],
+        "sigmas": [scheduler_id, 0],
+        "latent_image": [cond_id, 1],
+    })
+
+    # --- 解碼 / 輸出 --------------------------------------------------------
+    images_id = graph.add("VAEDecode", {
+        "samples": [sampled_id, 0],
+        "vae": [video_vae_id, 0],
+    })
+    audio_id = graph.add("VAEDecodeAudio", {
+        "samples": [sampled_id, 0],
+        "vae": [audio_vae_id, 0],
+    })
+    video_id = graph.add("CreateVideo", {
+        "images": [images_id, 0],
+        "fps": float(spec.fps),
+        "audio": [audio_id, 0],
+        "bit_depth": int(spec.bit_depth),
+    })
+    graph.add("SaveVideo", {
+        "video": [video_id, 0],
+        "filename_prefix": spec.filename_prefix,
+        "format": spec.video_format,
+        "codec": spec.video_codec,
+    })
+    return graph.nodes, notes
+
+
+def spec_from_profile(profile, prompt: str, **overrides) -> GenSpec:
+    """用算力檔案（Profile）產生一份預設 GenSpec。"""
+    spec = GenSpec(
+        prompt=prompt,
+        width=profile.width,
+        height=profile.height,
+        length=profile.length,
+        steps=profile.steps,
+        mode=profile.mode,
+        unet=profile.unet.filename,
+        clip=profile.clip.filename,
+        video_vae=profile.video_vae.filename,
+        audio_vae=profile.audio_vae.filename,
+        turbo_lora=profile.turbo_lora.filename if profile.turbo_lora else None,
+        turbo_lowvram=profile.lowvram,
+        sol_attn=profile.sol_attn,
+        ffn_chunks=profile.ffn_chunks,
+    )
+    for key, value in overrides.items():
+        setattr(spec, key, value)
+    return spec
+

In [ ]:
%%writefile h3colab/client.py
+"""ComfyUI HTTP API 客戶端：提交、輪詢、收結果、診斷錯誤。"""
+
+from __future__ import annotations
+
+import json
+import os
+import shutil
+import time
+import uuid
+from pathlib import Path
+
+
+class ComfyError(RuntimeError):
+    """ComfyUI 回報的錯誤（節點級錯誤會被整理成可讀訊息）。"""
+
+
+class ComfyClient:
+    """極簡 ComfyUI 客戶端（只用 requests）。
+
+    input_dir：ComfyUI 的 input/ 目錄。**ComfyUI 沒有 /upload/audio 端點**
+    （只有 /upload/image 與 /upload/mask），所以參考音訊必須直接放進
+    input/ 目錄 —— Colab 上 ComfyUI 與本檔同機器，直接複製即可。
+    """
+
+    def __init__(
+        self,
+        base_url: str = "http://127.0.0.1:8188",
+        timeout: int = 60,
+        input_dir: str | os.PathLike | None = None,
+    ) -> None:
+        self.base_url = base_url.rstrip("/")
+        self.timeout = timeout
+        self.client_id = str(uuid.uuid4())
+        self.input_dir = str(input_dir) if input_dir else None
+
+    # --- 低階 ---------------------------------------------------------------
+
+    def _get(self, path: str, **params):
+        import requests
+
+        response = requests.get(f"{self.base_url}{path}", params=params, timeout=self.timeout)
+        if response.status_code >= 400:
+            raise ComfyError(f"GET {path} -> HTTP {response.status_code}: {response.text[:300]}")
+        return response.json()
+
+    def _post(self, path: str, payload: dict):
+        import requests
+
+        response = requests.post(
+            f"{self.base_url}{path}", json=payload, timeout=self.timeout
+        )
+        if response.status_code >= 400:
+            raise ComfyError(f"POST {path} -> HTTP {response.status_code}: {response.text[:600]}")
+        return response.json()
+
+    # --- 查詢 ---------------------------------------------------------------
+
+    def system_stats(self) -> dict:
+        return self._get("/system_stats")
+
+    def object_info(self) -> dict:
+        return self._get("/object_info")
+
+    def available_nodes(self) -> set[str]:
+        return set(self.object_info().keys())
+
+    def queue_status(self) -> dict:
+        return self._get("/queue")
+
+    def history(self, prompt_id: str) -> dict:
+        return self._get(f"/history/{prompt_id}")
+
+    # --- 上傳 ---------------------------------------------------------------
+
+    def _upload(self, endpoint: str, field: str, path, subfolder: str, mime: str) -> str:
+        import requests
+
+        with open(path, "rb") as handle:
+            response = requests.post(
+                f"{self.base_url}/upload/{endpoint}",
+                files={field: (Path(path).name, handle, mime)},
+                data={"type": "input", "overwrite": "true", "subfolder": subfolder},
+                timeout=self.timeout,
+            )
+        if response.status_code >= 400:
+            raise ComfyError(f"上傳失敗 HTTP {response.status_code}: {response.text[:300]}")
+        data = response.json()
+        name = data.get("name")
+        if data.get("subfolder"):
+            name = f"{data['subfolder']}/{name}"
+        return name
+
+    def upload_image(self, path: str | os.PathLike, subfolder: str = "") -> str:
+        """把本機圖片上傳到 ComfyUI 的 input/，回傳可餵給節點的檔名。"""
+        return self._upload("image", "image", path, subfolder, "image/png")
+
+    def upload_audio(self, path: str | os.PathLike, subfolder: str = "") -> str:
+        """把音訊放進 ComfyUI 的 input/（ComfyUI 沒有音訊上傳端點）。
+
+        回傳可直接餵給節點的檔名（相對 input/）。
+        """
+        if not self.input_dir:
+            raise ComfyError(
+                "ComfyUI 沒有 /upload/audio 端點（只有 image/mask），參考音訊必須"
+                "直接放進 input/ 目錄。請用 ComfyClient(input_dir=...) 指定。"
+            )
+        source = Path(path)
+        dest_dir = Path(self.input_dir) / subfolder if subfolder else Path(self.input_dir)
+        dest_dir.mkdir(parents=True, exist_ok=True)
+        dest = dest_dir / source.name
+        index = 1
+        while dest.exists():
+            dest = dest_dir / f"{source.stem}_{index}{source.suffix}"
+            index += 1
+        shutil.copy2(source, dest)
+        return f"{subfolder}/{dest.name}" if subfolder else dest.name
+
+    # --- 執行 ---------------------------------------------------------------
+
+    def submit(self, graph: dict) -> str:
+        payload = {"prompt": graph, "client_id": self.client_id}
+        data = self._post("/prompt", payload)
+        prompt_id = data.get("prompt_id")
+        if not prompt_id:
+            raise ComfyError(f"ComfyUI 沒給 prompt_id：{json.dumps(data)[:400]}")
+        return prompt_id
+
+    def wait(self, prompt_id: str, timeout: int = 1800, poll: int = 3) -> dict:
+        """等到工作完成，回傳 history 條目；失敗會拋 ComfyError。"""
+        deadline = time.time() + timeout
+        while time.time() < deadline:
+            data = self.history(prompt_id)
+            entry = data.get(prompt_id)
+            if entry:
+                status = entry.get("status", {})
+                errors = [
+                    m for m in status.get("messages", [])
+                    if isinstance(m, list) and m and m[0] == "execution_error"
+                ]
+                if errors:
+                    detail = errors[0][1]
+                    raise ComfyError(
+                        "ComfyUI 執行失敗："
+                        f"{detail.get('node_type')} 節點 {detail.get('node_id')} "
+                        f"{detail.get('exception_type')}: {detail.get('exception_message')}"
+                    )
+                if entry.get("outputs"):
+                    return entry
+            time.sleep(poll)
+        raise ComfyError(f"等待 {prompt_id} 超時（{timeout}s）")
+
+    def download_outputs(self, entry: dict, dest_dir: str | os.PathLike) -> list[str]:
+        """把產出的影片抓回本機，回傳檔案路徑清單。"""
+        import requests
+
+        dest = Path(dest_dir)
+        dest.mkdir(parents=True, exist_ok=True)
+        saved: list[str] = []
+        for node_output in (entry.get("outputs") or {}).values():
+            for kind in ("videos", "images", "gifs", "audio"):
+                for item in node_output.get(kind, []) or []:
+                    filename = item.get("filename")
+                    if not filename:
+                        continue
+                    subfolder = item.get("subfolder", "")
+                    file_type = item.get("type", "output")
+                    url = (
+                        f"{self.base_url}/view?filename={filename}"
+                        f"&subfolder={subfolder}&type={file_type}"
+                    )
+                    response = requests.get(url, timeout=self.timeout)
+                    if response.status_code >= 400:
+                        continue
+                    target = dest / subfolder / filename if subfolder else dest / filename
+                    target.parent.mkdir(parents=True, exist_ok=True)
+                    target.write_bytes(response.content)
+                    saved.append(str(target))
+        return saved
+
+    def run(self, graph: dict, timeout: int = 1800, dest_dir: str | os.PathLike | None = None):
+        """提交 → 等完成 → （可選）抓檔。回傳 (history_entry, saved_paths)。"""
+        prompt_id = self.submit(graph)
+        entry = self.wait(prompt_id, timeout=timeout)
+        saved = self.download_outputs(entry, dest_dir) if dest_dir else []
+        return entry, saved

In [ ]:
%%writefile h3colab/tts_voice.py
+"""TTS：中文語音 → 32kHz 單聲道 wav，並量出**真實音軌長度**。
+
+為什麼要量長度：聲音驅動的數字人，畫面時間碼必須跟語速一致，
+不然嘴型對不上、就會「亂講話」。這裡把每句台詞的實際秒數回報出來，
+交給 prompt_compiler.retime_beats() 回填時間碼。
+
+後端（依序嘗試）：
+* ``edge``  —— edge-tts，免費、無需金鑰，繁中音色最齊
+* ``gtts``  —— gTTS，免費備援（音色較機械）
+"""
+
+from __future__ import annotations
+
+import asyncio
+import os
+import shutil
+import struct
+import subprocess
+import wave
+from dataclasses import dataclass
+from pathlib import Path
+
+SAMPLE_RATE = 32000  # H3 音訊 VAE 的原生採樣率
+
+VOICES = {
+    "zh-TW-HsiaoChenNeural": "台灣女聲（自然、播報感）",
+    "zh-TW-HsiaoYuNeural": "台灣女聲（年輕）",
+    "zh-TW-YunxiNeural": "台灣男聲（年輕、活潑）",
+    "zh-TW-YunYangNeural": "台灣男聲（播報）",
+    "zh-CN-XiaoxiaoNeural": "中國女聲",
+    "zh-CN-YunxiNeural": "中國男聲",
+}
+
+DEFAULT_VOICE = "zh-TW-HsiaoChenNeural"
+
+
+@dataclass
+class ScriptLine:
+    """一句台詞 + 它的音檔 + 真實長度。"""
+
+    text: str
+    path: str
+    duration: float
+
+
+class TTSError(RuntimeError):
+    pass
+
+
+# --- 音訊工具 ---------------------------------------------------------------
+
+def to_wav(src: str | os.PathLike, dst: str | os.PathLike, sample_rate: int = SAMPLE_RATE) -> str:
+    """轉成 32kHz 單聲道 16-bit wav（H3 音訊 VAE 要這個格式）。"""
+    src, dst = Path(src), Path(dst)
+    dst.parent.mkdir(parents=True, exist_ok=True)
+    ffmpeg = shutil.which("ffmpeg")
+    if not ffmpeg:
+        raise TTSError(
+            "找不到 ffmpeg：Colab 先跑 !apt-get install -y ffmpeg，"
+            "或用 !pip install imageio-ffmpeg 拿一份。"
+        )
+    cmd = [
+        ffmpeg, "-y", "-i", str(src),
+        "-ac", "1", "-ar", str(sample_rate), "-acodec", "pcm_s16le",
+        str(dst),
+    ]
+    result = subprocess.run(cmd, capture_output=True)
+    if result.returncode != 0:
+        raise TTSError(f"ffmpeg 轉檔失敗：{result.stderr.decode('utf-8', 'ignore')[-400:]}")
+    return str(dst)
+
+
+def probe_duration(path: str | os.PathLike) -> float:
+    """量音檔長度（wav 用 wave 模組；其他格式走 ffprobe）。"""
+    path = Path(path)
+    if path.suffix.lower() == ".wav":
+        with wave.open(str(path), "rb") as handle:
+            frames = handle.getnframes()
+            rate = handle.getframerate() or SAMPLE_RATE
+            return round(frames / float(rate), 3)
+    ffprobe = shutil.which("ffprobe")
+    if ffprobe:
+        result = subprocess.run(
+            [ffprobe, "-v", "error", "-show_entries", "format=duration",
+             "-of", "default=noprint_wrappers=1:nokey=1", str(path)],
+            capture_output=True, text=True,
+        )
+        if result.returncode == 0 and result.stdout.strip():
+            try:
+                return round(float(result.stdout.strip()), 3)
+            except ValueError:
+                pass
+    raise TTSError(f"量不到音檔長度：{path}")
+
+
+def concat_wavs(paths: list[str], dst: str | os.PathLike) -> str:
+    """把多段 wav 串成一條（純 PCM 串接，不重採樣）。"""
+    if not paths:
+        raise TTSError("沒有要串接的音檔")
+    dst = Path(dst)
+    dst.parent.mkdir(parents=True, exist_ok=True)
+    with wave.open(str(paths[0]), "rb") as first:
+        params = first.getparams()
+        frames = [first.readframes(first.getnframes())]
+    for path in paths[1:]:
+        with wave.open(str(path), "rb") as handle:
+            if handle.getnchannels() != params.nchannels or \
+               handle.getframerate() != params.framerate:
+                raise TTSError("串接的音檔格式不一致，請先統一成 32kHz 單聲道")
+            frames.append(handle.readframes(handle.getnframes()))
+    with wave.open(str(dst), "wb") as out:
+        out.setparams(params)
+        for chunk in frames:
+            out.writeframes(chunk)
+    return str(dst)
+
+
+# --- 合成 -------------------------------------------------------------------
+
+async def _edge_synth(text: str, voice: str, out_path: str, rate: str, pitch: str) -> None:
+    import edge_tts
+
+    communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
+    await communicate.save(out_path)
+
+
+async def _gtts_synth(text: str, out_path: str, lang: str = "zh-TW") -> None:
+    from gtts import gTTS
+
+    await asyncio.to_thread(gTTS(text=text, lang=lang).save, out_path)
+
+
+def synthesize(
+    text: str,
+    out_path: str | os.PathLike,
+    voice: str = DEFAULT_VOICE,
+    rate: str = "+0%",
+    pitch: str = "+0Hz",
+    backend: str = "edge",
+) -> str:
+    """把一句話合成成 wav（32kHz 單聲道），回傳路徑。"""
+    text = (text or "").strip()
+    if not text:
+        raise TTSError("空字串")
+    out_path = Path(out_path)
+    out_path.parent.mkdir(parents=True, exist_ok=True)
+    raw = out_path.with_suffix(".raw.mp3" if backend == "edge" else ".raw.mp3")
+
+    try:
+        if backend == "edge":
+            asyncio.run(_edge_synth(text, voice, str(raw), rate, pitch))
+        elif backend == "gtts":
+            asyncio.run(_gtts_synth(text, str(raw)))
+        else:
+            raise TTSError(f"未知 TTS 後端：{backend}")
+    except ImportError as exc:
+        raise TTSError(
+            f"缺套件：{exc}。Colab 請跑 !pip install edge-tts"
+            "（備援：!pip install gTTS 後用 backend='gtts'）"
+        ) from exc
+    except Exception as exc:  # noqa: BLE001
+        raise TTSError(f"TTS 失敗：{exc}") from exc
+
+    to_wav(raw, out_path)
+    raw.unlink(missing_ok=True)
+    return str(out_path)
+
+
+def synth_script(
+    lines: list[str],
+    out_dir: str | os.PathLike,
+    voice: str = DEFAULT_VOICE,
+    rate: str = "+0%",
+    pitch: str = "+0Hz",
+    backend: str = "edge",
+) -> list[ScriptLine]:
+    """逐句合成，回傳每句的音檔路徑與真實長度。
+
+    句與句之間不插入停頓（H3 會自行處理韻律）；要停頓請在文字裡加標點。
+    """
+    out_dir = Path(out_dir)
+    out_dir.mkdir(parents=True, exist_ok=True)
+    result: list[ScriptLine] = []
+    for index, text in enumerate(lines):
+        text = (text or "").strip()
+        if not text:
+            continue
+        path = out_dir / f"line_{index:02d}.wav"
+        synthesize(text, path, voice=voice, rate=rate, pitch=pitch, backend=backend)
+        result.append(ScriptLine(text=text, path=str(path), duration=probe_duration(path)))
+    return result
+
+
+def list_voices() -> dict[str, str]:
+    """本專案推薦的音色表。"""
+    return dict(VOICES)
+
+
+def install_hint() -> str:
+    return (
+        "Colab 安裝：!pip install -q edge-tts && !apt-get install -y ffmpeg\n"
+        "（Edge TTS 是微軟線上服務，免費但需要網路；離線可改用 gTTS）"
+    )

In [ ]:
%%writefile h3colab/setup_colab.py
+"""Colab 佈署：冪等安裝 ComfyUI、節點、權重（雲端硬碟當倉庫）。
+
+設計原則
+--------
+1. **冪等**：每一步都先檢查再做事，重跑不會重複下載、重複 clone。
+2. **倉庫與執行分離**：權重可以存在 Google Drive（5TB 慢慢長），執行前再
+   複製一份到本機磁碟（Drive 的 FUSE 讀取很慢，直接從 Drive 載入模型會
+   慢到不能忍）。``--store local`` 則是單次直落本機，最快但不留檔。
+3. **依算力選權重**：24GB 卡用 w6a8 + 4-bit CLIP，40GB 卡用 int8。
+"""
+
+from __future__ import annotations
+
+import os
+import shutil
+import subprocess
+import sys
+from pathlib import Path
+
+from .config import (
+    COMFY_ROOT, DRIVE_ROOT, GIT_DIRECTOR, GIT_SOL_ATTN, GIT_TURBO,
+    GIT_VIDEO_HELPER, ModelFile, Profile,
+)
+
+NODE_REPOS = {
+    "ComfyUI-MiniMax-H3-Turbo": GIT_TURBO,
+    "ComfyUI-sol-attn": GIT_SOL_ATTN,
+    "ComfyUI_MiniMaxH3_Director": GIT_DIRECTOR,
+    "ComfyUI-VideoHelperSuite": GIT_VIDEO_HELPER,
+}
+
+REQUIRED_NODES = ("ComfyUI-MiniMax-H3-Turbo",)
+OPTIONAL_NODES = ("ComfyUI-sol-attn", "ComfyUI_MiniMaxH3_Director", "ComfyUI-VideoHelperSuite")
+
+
+def log(message: str) -> None:
+    print(f"[h3colab] {message}", flush=True)
+
+
+def run(cmd: list[str], cwd: str | os.PathLike | None = None, check: bool = True) -> int:
+    log("$ " + " ".join(str(part) for part in cmd))
+    result = subprocess.run(cmd, cwd=str(cwd) if cwd else None)
+    if check and result.returncode != 0:
+        raise RuntimeError(f"指令失敗（{result.returncode}）：{' '.join(cmd)}")
+    return result.returncode
+
+
+# --- GPU 偵測 ---------------------------------------------------------------
+
+def detect_gpu() -> dict:
+    """回傳 {'name', 'vram_gb', 'capability', 'available'}。"""
+    try:
+        import torch
+
+        if not torch.cuda.is_available():
+            return {"name": "none", "vram_gb": 0.0, "capability": None, "available": False}
+        index = torch.cuda.current_device()
+        props = torch.cuda.get_device_properties(index)
+        return {
+            "name": props.name,
+            "vram_gb": round(props.total_memory / (1024 ** 3), 1),
+            "capability": f"{props.major}{props.minor}",
+            "available": True,
+        }
+    except Exception as exc:  # noqa: BLE001
+        log(f"GPU 偵測失敗：{exc}")
+        return {"name": "unknown", "vram_gb": 0.0, "capability": None, "available": False}
+
+
+def sol_attn_supported(capability: str | None) -> bool:
+    """sol-attn 支援的架構：SM86 / SM89 / SM90 / SM100 / SM120 / SM121。
+
+    L4 = SM89 ✅、A100 = SM80 ❌、T4 = SM75 ❌。
+    """
+    if not capability:
+        return False
+    return capability in {"86", "89", "90", "100", "120", "121"}
+
+
+# --- ComfyUI ----------------------------------------------------------------
+
+def ensure_comfyui(root: str | os.PathLike = COMFY_ROOT) -> Path:
+    """確認 ComfyUI 在位（H3 需要 >= 0.30.0）。"""
+    root = Path(root)
+    if (root / "main.py").exists():
+        log(f"ComfyUI 已在 {root}")
+    else:
+        log(f"clone ComfyUI -> {root}")
+        root.parent.mkdir(parents=True, exist_ok=True)
+        run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git",
+             str(root)])
+    ensure_torch(root)
+    requirements = root / "requirements.txt"
+    if requirements.exists():
+        log("安裝 ComfyUI requirements")
+        run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
+    return root
+
+
+def ensure_torch(root: str | os.PathLike = COMFY_ROOT) -> None:
+    """保證 torch 有 CUDA；Colab 內建 torch 通常已可用。"""
+    try:
+        import torch
+
+        if torch.cuda.is_available():
+            log(f"torch {torch.__version__} 已可用 CUDA")
+            return
+    except Exception:  # noqa: BLE001
+        pass
+    log("安裝 CUDA 版 torch（cu124）")
+    run([
+        sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio",
+        "--index-url", "https://download.pytorch.org/whl/cu124",
+    ])
+
+
+def ensure_nodes(root: str | os.PathLike = COMFY_ROOT, include_optional: bool = True) -> dict[str, str]:
+    """clone 需要的 custom nodes（已存在就 git pull）。"""
+    custom = Path(root) / "custom_nodes"
+    custom.mkdir(parents=True, exist_ok=True)
+    wanted = list(REQUIRED_NODES) + (list(OPTIONAL_NODES) if include_optional else [])
+    result: dict[str, str] = {}
+    for name in wanted:
+        url = NODE_REPOS[name]
+        path = custom / name
+        if (path / ".git").exists():
+            log(f"更新 {name}")
+            run(["git", "-C", str(path), "pull", "--ff-only"], check=False)
+        else:
+            log(f"clone {name}")
+            if not run(["git", "clone", "--depth", "1", url, str(path)], check=False):
+                result[name] = "FAILED"
+                continue
+        req = path / "requirements.txt"
+        if req.exists():
+            run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)
+        result[name] = str(path)
+    return result
+
+
+# --- 權重 -------------------------------------------------------------------
+
+def _hf_download(model: ModelFile, dest_dir: Path) -> Path:
+    """用 huggingface_hub 下載單一檔案到 dest_dir（支援續傳）。"""
+    from huggingface_hub import hf_hub_download
+
+    dest_dir.mkdir(parents=True, exist_ok=True)
+    log(f"下載 {model.filename}（{model.size_gb} GB）← {model.repo}")
+    path = hf_hub_download(
+        repo_id=model.repo,
+        filename=model.filename,
+        local_dir=str(dest_dir),
+    )
+    return Path(path)
+
+
+def store_root(store: str) -> Path:
+    """store='drive' → 雲端硬碟倉庫；store='local' → 本機磁碟。"""
+    if store == "drive":
+        return Path(DRIVE_ROOT)
+    if store == "local":
+        return Path(COMFY_ROOT)
+    raise ValueError(f"未知 store：{store}")
+
+
+def download_model(
+    model: ModelFile,
+    store: str = "drive",
+    materialize: bool = True,
+    root: str | os.PathLike = COMFY_ROOT,
+) -> Path:
+    """下載一個權重檔，回傳它在 ComfyUI models/ 底下的最終路徑。"""
+    target_dir = Path(COMFY_ROOT) / "models" / model.subdir
+    final = target_dir / model.filename
+    if final.exists() and final.stat().st_size > 0:
+        log(f"已存在，略過：{final.name}")
+        return final
+
+    warehouse = store_root(store) / "models" / model.subdir
+    downloaded = _hf_download(model, warehouse)
+
+    if materialize:
+        target_dir.mkdir(parents=True, exist_ok=True)
+        if downloaded.resolve() != final.resolve():
+            log(f"複製到執行目錄：{final.name}（Drive 讀取慢，複製後才跑得動）")
+            shutil.copy2(downloaded, final)
+        return final
+    return downloaded
+
+
+def download_profile(
+    profile: Profile,
+    store: str = "drive",
+    materialize: bool = True,
+    extra: tuple[ModelFile, ...] = (),
+) -> list[Path]:
+    """把一個 profile 需要的權重全部備齊。"""
+    paths: list[Path] = []
+    for model in list(profile.models) + list(extra):
+        paths.append(download_model(model, store=store, materialize=materialize))
+    return paths
+
+
+def model_report(profile: Profile, root: str | os.PathLike = COMFY_ROOT) -> list[dict]:
+    """檢查權重是否就位。"""
+    rows: list[dict] = []
+    for model in profile.models:
+        path = Path(root) / "models" / model.subdir / model.filename
+        rows.append({
+            "file": model.filename,
+            "subdir": model.subdir,
+            "size_gb": model.size_gb,
+            "present": path.exists(),
+            "path": str(path),
+        })
+    return rows
+
+
+# --- ComfyUI 啟動 -----------------------------------------------------------
+
+def start_comfyui(
+    root: str | os.PathLike = COMFY_ROOT,
+    port: int = 8188,
+    lowvram: bool = False,
+    extra_args: list[str] | None = None,
+    log_path: str | os.PathLike | None = None,
+) -> subprocess.Popen:
+    """背景啟動 ComfyUI，回傳 Popen（呼叫端要自己收掉）。"""
+    root = Path(root)
+    log_path = Path(log_path or (root / "logs" / "comfyui_colab.log"))
+    log_path.parent.mkdir(parents=True, exist_ok=True)
+    cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(port)]
+    if lowvram:
+        cmd.append("--lowvram")
+    cmd.extend(extra_args or [])
+    out = open(log_path, "ab")
+    log(f"啟動 ComfyUI：{' '.join(cmd)}")
+    return subprocess.Popen(
+        cmd, cwd=str(root), stdout=out, stderr=subprocess.STDOUT,
+        start_new_session=True,
+    )
+
+
+def wait_ready(base_url: str = "http://127.0.0.1:8188", timeout: int = 180) -> bool:
+    """等 ComfyUI 起來。"""
+    import time
+
+    import requests
+
+    deadline = time.time() + timeout
+    while time.time() < deadline:
+        try:
+            response = requests.get(f"{base_url}/system_stats", timeout=5)
+            if response.status_code == 200:
+                return True
+        except Exception:  # noqa: BLE001
+            pass
+        time.sleep(3)
+    return False

In [ ]:
%%writefile h3colab/cli.py
+"""命令列入口：doctor / setup / prompt / tts / generate / video。
+
+    python -m h3colab.cli doctor
+    python -m h3colab.cli setup --profile l4 --download --store drive
+    python -m h3colab.cli prompt --brief "..." --backend llm
+    python -m h3colab.cli tts --script script.txt --out out/tts
+    python -m h3colab.cli video --script script.txt --profile l4_ref \
+        --subject "a Taiwanese news anchor" --scene "a modern studio desk"
+"""
+
+from __future__ import annotations
+
+import argparse
+import json
+import os
+import sys
+from pathlib import Path
+
+from . import config, payloads, prompt_compiler, tts_voice
+from .client import ComfyClient, ComfyError
+
+
+def _print(text: str) -> None:
+    print(text, flush=True)
+
+
+def _client(base_url: str) -> ComfyClient:
+    """建立客戶端（附上 ComfyUI 的 input/ 目錄，參考音訊要直接放進去）。"""
+    return ComfyClient(base_url, input_dir=os.path.join(config.COMFY_ROOT, "input"))
+
+
+# --- doctor -----------------------------------------------------------------
+
+def cmd_doctor(args: argparse.Namespace) -> int:
+    from .setup_colab import detect_gpu, model_report, sol_attn_supported
+
+    gpu = detect_gpu()
+    _print("== GPU ==")
+    _print(json.dumps(gpu, ensure_ascii=False))
+    if gpu["available"]:
+        _print(f"   sol-attn 可用：{sol_attn_supported(gpu['capability'])}（SM{gpu['capability']}）")
+    else:
+        _print("   [注意] 偵測不到 CUDA（沒裝 torch 或沒有 GPU）。"
+               "下面的 profile 建議不可靠，請手動指定 --profile。")
+    profile = config.pick_profile(gpu["vram_gb"], needs_ref=args.ref)
+    _print(f"   建議 profile：{profile.key} — {profile.label}")
+    if profile.warn:
+        _print(f"   注意：{profile.warn}")
+
+    _print("\n== ComfyUI ==")
+    root = Path(config.COMFY_ROOT)
+    _print(f"   {root} {'存在' if (root / 'main.py').exists() else '尚未安裝'}")
+    for name in (
+        "ComfyUI-MiniMax-H3-Turbo",
+        "ComfyUI-sol-attn",
+        "ComfyUI_MiniMaxH3_Director",
+    ):
+        path = root / "custom_nodes" / name
+        _print(f"   {name}: {'已安裝' if path.exists() else '未安裝'}")
+
+    _print("\n== 權重 ==")
+    for row in model_report(profile):
+        mark = "OK " if row["present"] else "缺 "
+        _print(f"   {mark}{row['file']}  ({row['size_gb']} GB)")
+
+    _print("\n== TTS ==")
+    _print("   " + tts_voice.install_hint())
+    return 0
+
+
+# --- setup ------------------------------------------------------------------
+
+def cmd_setup(args: argparse.Namespace) -> int:
+    from .setup_colab import (
+        download_profile, ensure_comfyui, ensure_nodes, log,
+    )
+
+    profile = config.PROFILES[args.profile]
+    log(f"profile：{profile.label}")
+    if profile.warn:
+        log(f"注意：{profile.warn}")
+
+    ensure_comfyui(config.COMFY_ROOT)
+    ensure_nodes(config.COMFY_ROOT, include_optional=not args.minimal)
+
+    if args.download:
+        download_profile(profile, store=args.store, materialize=not args.no_copy)
+    else:
+        log("略過權重下載（加 --download 才下）")
+    return 0
+
+
+# --- prompt -----------------------------------------------------------------
+
+def _brief_from_args(args: argparse.Namespace) -> prompt_compiler.Brief:
+    beats = list(args.beat or [])
+    if args.script:
+        script_path = Path(args.script)
+        text = script_path.read_text(encoding="utf-8")
+        beats.extend([line.strip() for line in text.splitlines() if line.strip()])
+    return prompt_compiler.Brief(
+        subject=args.subject or "",
+        scene=args.scene or "",
+        beats=beats,
+        look=args.look or "",
+        camera=args.camera or "",
+        audio=args.audio,
+        voice_driven=args.voice_driven,
+        duration=args.duration,
+        template=args.template,
+        identity_lock=args.identity or "",
+    )
+
+
+def cmd_prompt(args: argparse.Namespace) -> int:
+    brief = _brief_from_args(args)
+    llm_config = prompt_compiler.LLMConfig.from_env() if args.backend == "llm" else None
+    text = prompt_compiler.enhance(
+        brief,
+        backend=args.backend,
+        llm_config=llm_config,
+        base_url=args.base_url,
+        task_type=args.task,
+    )
+    if args.out:
+        Path(args.out).parent.mkdir(parents=True, exist_ok=True)
+        Path(args.out).write_text(text, encoding="utf-8")
+        _print(f"已寫入 {args.out}")
+    _print(text)
+    return 0
+
+
+# --- tts --------------------------------------------------------------------
+
+def cmd_tts(args: argparse.Namespace) -> int:
+    out_dir = Path(args.out)
+    out_dir.mkdir(parents=True, exist_ok=True)
+    if args.script:
+        lines = [
+            line.strip()
+            for line in Path(args.script).read_text(encoding="utf-8").splitlines()
+            if line.strip()
+        ]
+    else:
+        lines = [args.text or ""]
+    lines = [line for line in lines if line]
+    if not lines:
+        _print("沒有台詞")
+        return 1
+    result = tts_voice.synth_script(
+        lines, out_dir, voice=args.voice, rate=args.rate, backend=args.backend
+    )
+    total = sum(item.duration for item in result)
+    manifest = {
+        "voice": args.voice,
+        "total_seconds": round(total, 3),
+        "lines": [
+            {"text": item.text, "path": item.path, "duration": item.duration}
+            for item in result
+        ],
+    }
+    (out_dir / "manifest.json").write_text(
+        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
+    )
+    _print(json.dumps(manifest, ensure_ascii=False, indent=2))
+    return 0
+
+
+# --- generate ---------------------------------------------------------------
+
+def _build_spec(args: argparse.Namespace, prompt: str, profile: config.Profile):
+    overrides = {}
+    for key in ("width", "height", "length", "steps", "seed", "task"):
+        value = getattr(args, key, None)
+        if value is not None:
+            overrides[key] = value
+    spec = payloads.spec_from_profile(profile, prompt, **overrides)
+    if not getattr(args, "no_sol_attn", False):
+        spec.sol_attn = profile.sol_attn
+    return spec
+
+
+def cmd_generate(args: argparse.Namespace) -> int:
+    profile = config.PROFILES[args.profile]
+    prompt = Path(args.prompt_file).read_text(encoding="utf-8") if args.prompt_file else args.prompt
+    if not prompt:
+        _print("沒有提示詞（--prompt 或 --prompt-file）")
+        return 1
+
+    client = _client(args.base_url)
+    available = client.available_nodes()
+    spec = _build_spec(args, prompt, profile)
+
+    if args.ref_image:
+        spec.ref_images = [client.upload_image(path) for path in args.ref_image]
+    if args.ref_audio:
+        spec.ref_audios = [client.upload_audio(path) for path in args.ref_audio]
+    if spec.task == "r2v" and not spec.ref_images and not spec.ref_audios:
+        _print("提示：r2v 建議至少給一個參考圖（--ref-image）或參考音訊（--ref-audio）")
+
+    spec.filename_prefix = args.prefix or f"video/{profile.key}"
+    graph, notes = payloads.build_graph(spec, available)
+    for note in notes:
+        _print(f"[注意] {note}")
+
+    _print(f"提交：{profile.label} / {spec.width}x{spec.height} / "
+           f"{spec.resolved_length()} 幀 / {spec.steps} 步 / {spec.mode}")
+    try:
+        entry, saved = client.run(graph, timeout=args.timeout, dest_dir=args.out)
+    except ComfyError as exc:
+        _print(f"[失敗] {exc}")
+        return 1
+    for path in saved:
+        _print(f"完成：{path}")
+    if not saved:
+        _print("完成，但沒抓到檔案（看 ComfyUI output/ 目錄）")
+    return 0
+
+
+# --- video（端到端：台詞 → TTS → 提示組 → 聲音驅動生成） ---------------------
+
+def cmd_video(args: argparse.Namespace) -> int:
+    profile = config.PROFILES[args.profile]
+    out_dir = Path(args.out)
+    out_dir.mkdir(parents=True, exist_ok=True)
+
+    lines = [
+        line.strip()
+        for line in Path(args.script).read_text(encoding="utf-8").splitlines()
+        if line.strip()
+    ]
+    if not lines:
+        _print("台詞檔是空的")
+        return 1
+
+    # 1) TTS：逐句合成並量真實長度
+    script_lines = tts_voice.synth_script(
+        lines, out_dir / "tts", voice=args.voice, rate=args.rate, backend=args.tts_backend
+    )
+    durations = [item.duration for item in script_lines]
+    total = sum(durations)
+    _print(f"台詞 {len(script_lines)} 句，總長 {total:.2f}s")
+
+    # 2) 時間碼：依真實音軌長度切
+    #    分鏡描述應該是「英文視覺描述」；台詞是中文，走 TTS → <Audio 1>。
+    #    沒給 --beat 就用台詞頂替（LLM 後端會翻譯；純規則後端會出中文）。
+    visual = list(args.beat or [])
+    if not visual:
+        visual = [item.text for item in script_lines]
+        if args.backend == "rule" and prompt_compiler.has_cjk(" ".join(visual)):
+            _print("[注意] 沒給 --beat，分鏡會直接用中文台詞。"
+                   "H3 用 Qwen3-VL 能讀中文，但官方測試路徑是英文；"
+                   "建議補 --beat（英文視覺描述）或改用 --backend llm。")
+    if len(visual) != len(script_lines):
+        _print(f"[注意] --beat 有 {len(visual)} 個，時間碼只有 {len(script_lines)} 段；"
+               "多出來的節拍會對不上語速。")
+        visual = (visual + [item.text for item in script_lines])[:len(script_lines)]
+    beats = prompt_compiler.distribute_beats(visual, total)
+    timed = prompt_compiler.retime_beats(beats, durations)
+
+    # 3) 提示組
+    brief = prompt_compiler.Brief(
+        subject=args.subject or "",
+        scene=args.scene or "",
+        beats=visual,
+        look=args.look or "",
+        camera=args.camera or "",
+        audio=args.audio or "clear spoken voice",
+        voice_driven=True,
+        duration=total,
+        timed_beats=timed,
+    )
+    llm_config = prompt_compiler.LLMConfig.from_env() if args.backend == "llm" else None
+    prompt = prompt_compiler.enhance(
+        brief, backend=args.backend, llm_config=llm_config,
+        base_url=args.base_url, task_type="r2v",
+    )
+    (out_dir / "prompt.txt").write_text(prompt, encoding="utf-8")
+    _print("\n--- 提示組 ---\n" + prompt + "\n")
+
+    # 4) 合併音軌 + 放進 input/
+    merged = tts_voice.concat_wavs([item.path for item in script_lines], out_dir / "voice.wav")
+    client = _client(args.base_url)
+    audio_name = client.upload_audio(merged)
+
+    # 5) 幀數要對上音軌長度
+    frames = config.snap_length(int(total * 24) + 24)
+    spec = payloads.spec_from_profile(
+        profile, prompt, task="r2v", length=frames, ref_audios=[audio_name],
+    )
+    if args.ref_image:
+        spec.ref_images = [client.upload_image(path) for path in args.ref_image]
+    spec.filename_prefix = args.prefix or f"video/{profile.key}_avatar"
+
+    available = client.available_nodes()
+    graph, notes = payloads.build_graph(spec, available)
+    for note in notes:
+        _print(f"[注意] {note}")
+
+    _print(f"提交：{profile.label} / {spec.width}x{spec.height} / {frames} 幀 / "
+           f"{spec.steps} 步 / r2v + <Audio 1>")
+    try:
+        entry, saved = client.run(graph, timeout=args.timeout, dest_dir=out_dir)
+    except ComfyError as exc:
+        _print(f"[失敗] {exc}")
+        return 1
+    for path in saved:
+        _print(f"完成：{path}")
+    return 0
+
+
+# --- 入口 -------------------------------------------------------------------
+
+def build_parser() -> argparse.ArgumentParser:
+    parser = argparse.ArgumentParser(prog="h3colab", description="Colab ⇄ ComfyUI ⇄ MiniMax H3")
+    sub = parser.add_subparsers(dest="command", required=True)
+
+    p = sub.add_parser("doctor", help="檢查 GPU / ComfyUI / 節點 / 權重")
+    p.add_argument("--ref", action="store_true", help="以 r2v（聲音驅動）角度檢查")
+    p.set_defaults(func=cmd_doctor)
+
+    p = sub.add_parser("setup", help="安裝 ComfyUI + 節點 + 權重")
+    p.add_argument("--profile", default="l4", choices=sorted(config.PROFILES))
+    p.add_argument("--store", default="drive", choices=["drive", "local"])
+    p.add_argument("--download", action="store_true", help="連權重一起下（預設不下）")
+    p.add_argument("--no-copy", action="store_true", help="不要複製到執行目錄")
+    p.add_argument("--minimal", action="store_true", help="只裝必要的節點")
+    p.set_defaults(func=cmd_setup)
+
+    p = sub.add_parser("prompt", help="把 brief 編譯成 H3 提示組")
+    p.add_argument("--brief", default="")
+    p.add_argument("--subject", default="")
+    p.add_argument("--scene", default="")
+    p.add_argument("--beat", action="append")
+    p.add_argument("--script")
+    p.add_argument("--look", default="")
+    p.add_argument("--camera", default="")
+    p.add_argument("--audio", default=None)
+    p.add_argument("--voice-driven", action="store_true")
+    p.add_argument("--duration", type=float, default=5.0)
+    p.add_argument("--template", default="storyboard", choices=["storyboard", "identity_lock"])
+    p.add_argument("--identity", default="")
+    p.add_argument("--backend", default="rule", choices=["rule", "llm", "director"])
+    p.add_argument("--base-url", default="http://127.0.0.1:8188")
+    p.add_argument("--task", default="t2v")
+    p.add_argument("--out")
+    p.set_defaults(func=cmd_prompt)
+
+    p = sub.add_parser("tts", help="合成中文語音並量真實長度")
+    p.add_argument("--text")
+    p.add_argument("--script")
+    p.add_argument("--out", default="out/tts")
+    p.add_argument("--voice", default=tts_voice.DEFAULT_VOICE)
+    p.add_argument("--rate", default="+0%")
+    p.add_argument("--backend", default="edge", choices=["edge", "gtts"])
+    p.set_defaults(func=cmd_tts)
+
+    p = sub.add_parser("generate", help="用既有提示詞生成")
+    p.add_argument("--profile", default="l4", choices=sorted(config.PROFILES))
+    p.add_argument("--prompt")
+    p.add_argument("--prompt-file")
+    p.add_argument("--task", choices=["t2v", "i2v", "fl2v", "r2v"])
+    p.add_argument("--width", type=int)
+    p.add_argument("--height", type=int)
+    p.add_argument("--length", type=int)
+    p.add_argument("--steps", type=int)
+    p.add_argument("--seed", type=int)
+    p.add_argument("--ref-image", action="append")
+    p.add_argument("--ref-audio", action="append")
+    p.add_argument("--no-sol-attn", action="store_true")
+    p.add_argument("--prefix")
+    p.add_argument("--out", default="out")
+    p.add_argument("--base-url", default="http://127.0.0.1:8188")
+    p.add_argument("--timeout", type=int, default=3600)
+    p.set_defaults(func=cmd_generate)
+
+    p = sub.add_parser("video", help="端到端：台詞 → TTS → 提示組 → 聲音驅動數字人")
+    p.add_argument("--script", required=True)
+    p.add_argument("--profile", default="l4_ref", choices=sorted(config.PROFILES))
+    p.add_argument("--subject", default="")
+    p.add_argument("--scene", default="")
+    p.add_argument("--look", default="")
+    p.add_argument("--camera", default="")
+    p.add_argument("--audio", default="")
+    p.add_argument("--beat", action="append",
+                   help="英文視覺描述（與台詞一句對一句）；不給就用台詞頂替")
+    p.add_argument("--voice", default=tts_voice.DEFAULT_VOICE)
+    p.add_argument("--rate", default="+0%")
+    p.add_argument("--tts-backend", default="edge", choices=["edge", "gtts"])
+    p.add_argument("--backend", default="rule", choices=["rule", "llm", "director"])
+    p.add_argument("--ref-image", action="append")
+    p.add_argument("--prefix")
+    p.add_argument("--out", default="out/avatar")
+    p.add_argument("--base-url", default="http://127.0.0.1:8188")
+    p.add_argument("--timeout", type=int, default=3600)
+    p.set_defaults(func=cmd_video)
+
+    return parser
+
+
+def main(argv: list[str] | None = None) -> int:
+    parser = build_parser()
+    args = parser.parse_args(argv)
+    return args.func(args)
+
+
+if __name__ == "__main__":
+    raise SystemExit(main())

In [ ]:
%%writefile h3colab/__init__.py
+"""Colab ⇄ ComfyUI ⇄ MiniMax H3 的客戶端套件。
+
+模組地圖：
+
+* ``config``          —— 模型檔案來源、算力檔案（profile）
+* ``prompt_compiler`` —— 提示強化：brief → 正確的 H3 提示組
+* ``tts_voice``       —— 中文 TTS → 32kHz wav + 真實音軌長度
+* ``payloads``        —— ComfyUI API 節點圖建構
+* ``client``          —— ComfyUI HTTP 客戶端
+* ``setup_colab``     —— 冪等安裝 / 權重倉庫
+* ``cli``             —— 命令列入口
+"""
+
+from .config import (
+    ALL_MODELS, PROFILES, Profile, ModelFile, pick_profile, snap_length,
+)
+from .prompt_compiler import (
+    Beat, Brief, LLMConfig, compile_prompt, distribute_beats, enhance,
+    has_cjk, retime_beats,
+)
+from .payloads import GenSpec, build_graph, spec_from_profile
+from .client import ComfyClient, ComfyError
+
+__all__ = [
+    "ALL_MODELS", "PROFILES", "Profile", "ModelFile", "pick_profile", "snap_length",
+    "Beat", "Brief", "LLMConfig", "compile_prompt", "distribute_beats", "enhance",
+    "has_cjk", "retime_beats",
+    "GenSpec", "build_graph", "spec_from_profile",
+    "ComfyClient", "ComfyError",
+]

In [ ]:
# == Cell 3：安裝 ComfyUI + 節點 + 權重 ==
import sys
sys.path.insert(0, "/content")
from h3colab import config
from h3colab.setup_colab import (
    detect_gpu, ensure_comfyui, ensure_nodes, download_profile,
)

gpu = detect_gpu()
print("GPU:", gpu)
profile = config.pick_profile(gpu["vram_gb"], needs_ref=(PROFILE.endswith("_ref")))
if profile.key != PROFILE:
    print(f"提醒：偵測到 {gpu['vram_gb']}GB，建議用 profile={profile.key}（{profile.label}）")
print("採用：", profile.label)
if profile.warn:
    print("注意：", profile.warn)

ensure_comfyui(config.COMFY_ROOT)
ensure_nodes(config.COMFY_ROOT)
download_profile(profile, store=STORE, materialize=True)

# 聲音驅動走的是 ref2va（PRISM / ref2va），跟 t2v/i2v 的 fl2va 是不同權重，
# 所以要另外抓一份，不然 Cell 7 會找不到模型。
if VOICE_DRIVEN:
    ref_profile = config.pick_profile(gpu["vram_gb"], needs_ref=True)
    print("另外抓聲音驅動用的 ref2va 權重：", ref_profile.label)
    download_profile(ref_profile, store=STORE, materialize=True)

In [ ]:
# == Cell 4：啟動 ComfyUI ==
import sys, time
sys.path.insert(0, "/content")
from h3colab import config
from h3colab.setup_colab import start_comfyui, wait_ready

proc = start_comfyui(
    config.COMFY_ROOT,
    lowvram=(gpu["vram_gb"] < 28),
    extra_args=["--use-ck-attention"],
)
ok = wait_ready(timeout=240)
print("ComfyUI ready:", ok)
if not ok:
    log = f"{config.COMFY_ROOT}/logs/comfyui_colab.log"
    print(open(log, encoding="utf-8", errors="ignore").read()[-3000:])

In [ ]:
# == Cell 5：提示強化（把簡單 brief 變成正確的 H3 提示組） ==
import sys
sys.path.insert(0, "/content")
from h3colab.prompt_compiler import Brief, compile_prompt, enhance

brief = Brief(
    subject="a Taiwanese female news anchor in a navy blazer",
    scene="a modern broadcast studio, dark background, glowing desk",
    look="50mm lens, shallow depth of field, soft key light from the left",
    beats=["looks straight into the lens and opens her mouth to speak",
           "raises one hand and points at the screen behind her"],
    camera="locked tripod, medium shot",
    duration=5.0,
    audio="clear female Mandarin voice",
)
prompt = compile_prompt(brief)
print(prompt)

# 想用 LLM 做強化（中文 brief 也可以）：
#   os.environ["GOOGLE_API_KEY"] = "..."
#   prompt = enhance(brief, backend="llm")

In [ ]:
# == Cell 6：TTS（繁中）+ 用真實音軌長度回推時間碼 ==
import sys
sys.path.insert(0, "/content")
from h3colab.tts_voice import synth_script, concat_wavs
from h3colab.prompt_compiler import Brief, distribute_beats, retime_beats, compile_prompt

script = [
    "各位觀眾晚安，今天我們要聊聊生成式AI。",
    "這支影片完全是在雲端算力上生出來的。",
]
lines = synth_script(script, "/content/out/tts", voice="zh-TW-HsiaoChenNeural")
for line in lines:
    print(f"{line.duration:5.2f}s  {line.text}")

total = sum(line.duration for line in lines)
beats = distribute_beats([line.text for line in lines], total)
timed = retime_beats(beats, [line.duration for line in lines])
print("時間碼：", [b.render() for b in timed])

brief = Brief(
    subject="a Taiwanese female news anchor in a navy blazer",
    scene="a modern broadcast studio, dark background, glowing desk",
    beats=[line.text for line in lines],
    look="50mm lens, shallow depth of field, soft key light from the left",
    camera="locked tripod, medium shot",
    audio="clear female Mandarin voice",
    voice_driven=True,
    duration=total,
    timed_beats=timed,
)
voice_prompt = compile_prompt(brief)
print(voice_prompt)
import os
os.makedirs("/content/out", exist_ok=True)
open("/content/out/prompt.txt", "w", encoding="utf-8").write(voice_prompt)
concat_wavs([line.path for line in lines], "/content/out/voice.wav")

In [ ]:
# == Cell 7：生成（r2v + <Audio 1> = 聲音驅動） ==
import sys
sys.path.insert(0, "/content")
from h3colab import config, payloads
from h3colab.client import ComfyClient

client = ComfyClient("http://127.0.0.1:8188", input_dir=f"{config.COMFY_ROOT}/input")
available = client.available_nodes()

# 聲音驅動要走 ref2va UNET（PRISM 就是 ref2va merge）
profile = config.PROFILES["l4_ref"]
frames = config.snap_length(int(total * 24) + 24)
spec = payloads.spec_from_profile(profile, voice_prompt, task="r2v", length=frames, seed=42)
spec.ref_audios = [client.upload_audio("/content/out/voice.wav")]
# 有角色參考圖就掛上（identity 會穩很多）：
# spec.ref_images = [client.upload_image("/content/out/face.png")]
spec.filename_prefix = "video/colab_avatar"

graph, notes = payloads.build_graph(spec, available)
for note in notes:
    print("[注意]", note)
print(f"提交 {spec.width}x{spec.height} / {spec.resolved_length()} 幀 / {spec.steps} 步 / {spec.mode}")

entry, saved = client.run(graph, timeout=3600, dest_dir="/content/out")
for path in saved:
    print("完成：", path)

In [ ]:
# == Cell 8：存檔（Drive 會活著，/content 不會） ==
import os, shutil, glob

dest = "/content/drive/MyDrive/h3_out"
os.makedirs(dest, exist_ok=True)
for path in glob.glob("/content/out/**/*.mp4", recursive=True):
    shutil.copy(path, dest)
    print("已存：", os.path.join(dest, os.path.basename(path)))

## 算力與額度

* **L4（24GB）**：864×480（0.4MP）、124 幀（約 5 秒）、Turbo 8 步 → 大概 5 分鐘內出片。
* **A100（40GB）**：1344×768（1.0MP）、124 幀、官方 25 步 → 品質檔。
* **T4（16GB）**：SM75，sol-attn 不支援、H3 走 bf16 很慢 —— 不建議，省下點數選 L4。
* **sol-attn**：L4 = SM89 官方支援；A100 = SM80 **不在**支援清單（SM86/89/90/100/120/121），
  會自動 dense fallback，功能正常但沒有加速。
* 用完記得中斷 runtime，別讓 VM 掛著吃點數。